# MedBot - AI Medical Assistant

## Deep Learning with Python - Final Project

**Team:** C. Cai | Y. Liao | D. Liu | Y. Qian | X. Wang | Y. Wang

This notebook demonstrates the technical implementation of MedBot, a RAG-based medical assistant.

### Contents
1. Environment Setup
2. Deep Learning: Embedding Model
3. Embedding Visualization (t-SNE)
4. Vector Database & Retrieval
5. Retrieval Evaluation Metrics
6. RAG Pipeline Demo
7. Summary

---
## 1. Environment Setup

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
import chromadb
from dotenv import load_dotenv

# For visualization
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv('../.env')
print("✅ Environment loaded successfully!")

---
## 2. Deep Learning: Embedding Model

### Why Sentence Transformers?

We use **Sentence Transformers** based on the BERT architecture to convert text into dense vector representations (embeddings). This is a key deep learning component:

- **Model**: `all-MiniLM-L6-v2`
- **Architecture**: 6-layer Transformer (distilled from BERT)
- **Output**: 384-dimensional dense vectors
- **Training**: Trained on 1B+ sentence pairs for semantic similarity

In [ ]:
# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"📦 Model: all-MiniLM-L6-v2")
print(f"📐 Embedding dimension: {model.get_sentence_embedding_dimension()}")
print(f"🔢 Parameters: ~22M (6-layer Transformer)")

In [ ]:
# Demonstrate embedding
sample_texts = [
    "I have a headache and feel dizzy",
    "My head hurts and I'm feeling lightheaded",  # Semantically similar
    "What is the capital of France?"  # Semantically different
]

embeddings = model.encode(sample_texts)

print("Input texts:")
for i, text in enumerate(sample_texts):
    print(f"  [{i}] {text}")

print(f"\nEmbedding shape: {embeddings.shape}")
print(f"Each text → {embeddings.shape[1]}-dimensional vector")

In [ ]:
# Calculate cosine similarity
similarity_matrix = cosine_similarity(embeddings)

print("🔍 Cosine Similarity Matrix:")
print("="*50)
labels = ["Headache", "Head hurts", "Capital?"]
df_sim = pd.DataFrame(similarity_matrix, index=labels, columns=labels)
print(df_sim.round(3))

print("\n📊 Observation:")
print(f"  • 'Headache' ↔ 'Head hurts': {similarity_matrix[0,1]:.3f} (HIGH - semantically similar)")
print(f"  • 'Headache' ↔ 'Capital?':  {similarity_matrix[0,2]:.3f} (LOW - different topics)")

---
## 3. Embedding Visualization (t-SNE)

We use t-SNE to visualize how the embedding model groups similar medical texts together.

In [ ]:
# Prepare texts from different medical categories
medical_texts = {
    "Symptoms": [
        "I have a severe headache",
        "My head is pounding",
        "I feel dizzy and nauseous",
        "I have chest pain",
        "My stomach hurts after eating",
    ],
    "Medications": [
        "Ibuprofen is used for pain relief",
        "Take aspirin for headaches",
        "Metformin treats diabetes",
        "Lisinopril lowers blood pressure",
        "Omeprazole reduces stomach acid",
    ],
    "Conditions": [
        "Diabetes affects blood sugar levels",
        "Hypertension is high blood pressure",
        "Asthma causes breathing difficulty",
        "Arthritis causes joint inflammation",
        "Migraine is a severe headache disorder",
    ]
}

# Flatten and encode
all_texts = []
all_labels = []
for category, texts in medical_texts.items():
    all_texts.extend(texts)
    all_labels.extend([category] * len(texts))

all_embeddings = model.encode(all_texts)
print(f"Encoded {len(all_texts)} texts into {all_embeddings.shape[1]}-dim vectors")

In [ ]:
# Apply t-SNE for dimensionality reduction
tsne = TSNE(n_components=2, random_state=42, perplexity=5)
embeddings_2d = tsne.fit_transform(all_embeddings)

# Plot
plt.figure(figsize=(10, 7))
colors = {'Symptoms': '#e74c3c', 'Medications': '#3498db', 'Conditions': '#2ecc71'}

for category in medical_texts.keys():
    mask = [l == category for l in all_labels]
    plt.scatter(
        embeddings_2d[mask, 0], 
        embeddings_2d[mask, 1], 
        c=colors[category], 
        label=category, 
        s=100, 
        alpha=0.7
    )

plt.title('t-SNE Visualization of Medical Text Embeddings', fontsize=14)
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('embedding_tsne.png', dpi=150)
plt.show()

print("\n📊 Observation: Similar medical concepts cluster together!")

---
## 4. Vector Database & Retrieval

We use **ChromaDB** as our vector database for efficient similarity search.

In [ ]:
from src.retriever import retrieve, format_context, add_documents

# Initialize ChromaDB
chroma_client = chromadb.PersistentClient(path="../vectorstore")
print("✅ ChromaDB initialized")
print(f"📁 Existing collections: {[c.name for c in chroma_client.list_collections()]}")

In [ ]:
# Add demo documents for testing
demo_docs = [
    "Headache can be caused by tension, migraine, or dehydration. Common treatments include rest, hydration, and pain relievers like ibuprofen.",
    "Dizziness may indicate inner ear problems, low blood pressure, or dehydration. Seek medical attention if persistent or accompanied by other symptoms.",
    "Fever is often a sign of infection. Rest and fluids are recommended. See a doctor if fever exceeds 103°F (39.4°C) or lasts more than 3 days.",
    "Chest pain can be caused by heart conditions, acid reflux, or muscle strain. Seek immediate medical attention if pain is severe or accompanied by shortness of breath.",
    "Fatigue can result from lack of sleep, stress, anemia, or thyroid disorders. Persistent fatigue should be evaluated by a healthcare provider."
]

demo_metadata = [
    {"source": "MedQuAD", "category": "headache", "id": "demo_1"},
    {"source": "MedQuAD", "category": "dizziness", "id": "demo_2"},
    {"source": "MedQuAD", "category": "fever", "id": "demo_3"},
    {"source": "MedQuAD", "category": "chest_pain", "id": "demo_4"},
    {"source": "MedQuAD", "category": "fatigue", "id": "demo_5"}
]

add_documents("demo_eval", demo_docs, demo_metadata)
print(f"✅ Added {len(demo_docs)} documents to 'demo_eval' collection")

In [ ]:
# Demo retrieval
query = "I have a headache and feel dizzy"
results = retrieve(query, "demo_eval", top_k=3)

print(f"🔍 Query: '{query}'\n")
print("📄 Retrieved Documents (Top-3):")
print("=" * 60)

for i, (doc, meta, dist) in enumerate(zip(
    results['documents'][0], 
    results['metadatas'][0], 
    results['distances'][0]
)):
    similarity = 1 - dist  # Convert distance to similarity
    print(f"\n[{i+1}] Similarity: {similarity:.4f}")
    print(f"    Category: {meta.get('category', 'N/A')}")
    print(f"    Content: {doc[:100]}...")

---
## 5. Retrieval Evaluation Metrics

We evaluate our retrieval system using standard IR metrics:
- **Recall@K**: Fraction of relevant documents retrieved in top-K
- **MRR (Mean Reciprocal Rank)**: Average of 1/rank for first relevant result

In [ ]:
# Test queries with expected relevant categories
eval_data = [
    {"query": "My head hurts badly", "relevant": "headache"},
    {"query": "I feel lightheaded and unsteady", "relevant": "dizziness"},
    {"query": "I have a high temperature", "relevant": "fever"},
    {"query": "Pain in my chest area", "relevant": "chest_pain"},
    {"query": "I'm always tired and exhausted", "relevant": "fatigue"},
    {"query": "Migraine symptoms", "relevant": "headache"},
    {"query": "Vertigo and balance problems", "relevant": "dizziness"},
    {"query": "Body temperature is high", "relevant": "fever"},
]

print(f"📋 Evaluation dataset: {len(eval_data)} test queries")

In [ ]:
def evaluate_retrieval(eval_data, collection, k_values=[1, 3, 5]):
    """Evaluate retrieval quality with Recall@K and MRR."""
    results_table = []
    
    for item in eval_data:
        query = item['query']
        relevant = item['relevant']
        
        # Retrieve top-K documents
        results = retrieve(query, collection, top_k=max(k_values))
        retrieved_categories = [m.get('category') for m in results['metadatas'][0]]
        
        # Calculate metrics
        row = {'query': query[:30] + '...', 'relevant': relevant}
        
        # Recall@K for each K
        for k in k_values:
            top_k_cats = retrieved_categories[:k]
            row[f'Recall@{k}'] = 1 if relevant in top_k_cats else 0
        
        # MRR
        if relevant in retrieved_categories:
            rank = retrieved_categories.index(relevant) + 1
            row['RR'] = 1 / rank
        else:
            row['RR'] = 0
            
        results_table.append(row)
    
    return pd.DataFrame(results_table)

# Run evaluation
eval_df = evaluate_retrieval(eval_data, "demo_eval")
print("📊 Per-Query Results:")
print(eval_df.to_string(index=False))

In [ ]:
# Summary metrics
print("\n" + "=" * 50)
print("📈 EVALUATION SUMMARY")
print("=" * 50)

metrics = {
    'Recall@1': eval_df['Recall@1'].mean(),
    'Recall@3': eval_df['Recall@3'].mean(),
    'Recall@5': eval_df['Recall@5'].mean(),
    'MRR': eval_df['RR'].mean()
}

for metric, value in metrics.items():
    bar = '█' * int(value * 20) + '░' * (20 - int(value * 20))
    print(f"{metric:12} {bar} {value:.2%}")

print("\n✅ High Recall@3 and MRR indicate effective semantic retrieval!")

In [ ]:
# Visualize metrics
fig, ax = plt.subplots(figsize=(8, 5))

metric_names = list(metrics.keys())
metric_values = list(metrics.values())
colors = ['#3498db', '#2ecc71', '#9b59b6', '#e74c3c']

bars = ax.bar(metric_names, metric_values, color=colors, edgecolor='black', linewidth=1.2)

# Add value labels
for bar, val in zip(bars, metric_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{val:.1%}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylim(0, 1.15)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Retrieval Evaluation Metrics', fontsize=14, fontweight='bold')
ax.axhline(y=0.8, color='green', linestyle='--', alpha=0.5, label='Target (80%)')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('retrieval_metrics.png', dpi=150)
plt.show()

---
## 6. RAG Pipeline Demo

Complete Retrieval-Augmented Generation pipeline:

```
Query → Embedding → Vector Search → Context → LLM → Response
```

In [ ]:
from src.llm import get_response, build_messages
from src.prompts import SYMPTOM_PROMPT

def medbot_query(question: str, collection: str = "demo_eval", verbose: bool = True):
    """Complete RAG pipeline for MedBot."""
    if verbose:
        print(f"❓ Question: {question}\n")
    
    # Step 1: Retrieve
    if verbose:
        print("📥 Step 1: Retrieving relevant documents...")
    results = retrieve(question, collection, top_k=3)
    context = format_context(results)
    if verbose:
        print(f"   Found {len(results['documents'][0])} relevant documents\n")
    
    # Step 2: Build prompt
    if verbose:
        print("📝 Step 2: Building prompt with context...")
    messages = build_messages(SYMPTOM_PROMPT, question, context)
    
    # Step 3: Generate
    if verbose:
        print("🤖 Step 3: Generating response...\n")
    response = get_response(messages)
    
    if verbose:
        print("=" * 60)
        print("💬 MedBot Response:")
        print("=" * 60)
        print(response)
    
    return response

In [ ]:
# Demo 1: Symptom query
medbot_query("I have a bad headache and feel dizzy. What could be wrong?")

In [ ]:
# Demo 2: Another query
medbot_query("What should I do if I have a high fever?")

---
## 7. Summary

### Deep Learning Components

| Component | Technology | Purpose |
|-----------|------------|--------|
| **Text Embedding** | Sentence Transformers (BERT-based) | Convert text to semantic vectors |
| **Vector Search** | ChromaDB + Cosine Similarity | Efficient nearest-neighbor retrieval |
| **Language Model** | DeepSeek V3 API | Context-aware response generation |

### Key Results

- ✅ Embedding model effectively captures semantic similarity
- ✅ t-SNE visualization shows clear clustering by topic
- ✅ High Recall@K and MRR demonstrate effective retrieval
- ✅ RAG pipeline successfully combines retrieval with generation

### Architecture Diagram

```
┌─────────────────────────────────────────────────────────────┐
│                     MedBot RAG Pipeline                     │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  User Query ──▶ Sentence Transformer ──▶ 384-dim Vector    │
│                     (all-MiniLM-L6-v2)                      │
│                            │                                │
│                            ▼                                │
│                     ChromaDB Search                         │
│                    (Cosine Similarity)                      │
│                            │                                │
│                            ▼                                │
│                    Top-K Documents                          │
│                            │                                │
│                            ▼                                │
│              Context + Prompt Template                      │
│                            │                                │
│                            ▼                                │
│                    DeepSeek V3 API                          │
│                            │                                │
│                            ▼                                │
│                     Final Response                          │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
print("🎉 Demo completed!")
print("\n📚 For more information, see:")
print("   • README.md - Project overview")
print("   • docs/DATA_FORMAT.md - Data specifications")
print("   • src/ - Source code modules")